# ETL — Escala da Aquisição e Preço
**TCC — Preços e concentração de mercado em compras públicas de insumos hospitalares, 2009–2023**  
Mônica Anatalia Bezerra de Araujo — MBA em Data Science e Analytics para Operações, POLI USP PRO

**Pergunta:** o tamanho do lote adquirido associa-se ao preco unitario pago?

**Desenho:** todas as comparacoes sao feitas DENTRO do mesmo item no mesmo ano.
Para cada registro calcula-se o desvio (em log) do preco e da quantidade em
relacao a mediana daquele item naquele ano. Isso neutraliza o tipo de produto
e o momento, isolando a relacao entre volume e preco.

**Analises:**
1. Correlacao entre desvio de quantidade e desvio de preco
2. Desvio de preco por quintil de escala
3. Escala tipica e preco relativo por unidade federativa
4. Diferenca territorial CONTROLADA por faixa de escala
5. Variancia explicada: escala x UF, antes e depois do controle

**Natureza do resultado:** descritivo. A base nao informa a motivacao da compra,
de modo que a associacao observada nao permite distinguir ganho de negociacao em
volume de aquisicoes emergenciais de pequena quantidade.

## 1. Setup

In [ ]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────
!pip install polars pyarrow scipy --quiet
import polars as pl, numpy as np
from scipy.stats import spearmanr
from pathlib import Path

BASE        = Path(PASTA_DADOS)
PASTA_BPS   = BASE / 'Base Harmonizacao Campos'
ARQ_IPCA    = BASE / 'IPCA Tratado' / 'ipca_deflator.parquet'
PASTA_SAIDA = BASE / 'Base Escala'
ANO_INICIO, ANO_FIM = 2009, 2023
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

## 2. Base com desvios de preço e quantidade
> Recupera a UF ausente em 2012 pelo CNPJ da instituicao, como no ETL_TERRITORIAL,
> para que a analise por unidade federativa cubra toda a serie.

In [ ]:
def n14(c):
    return pl.col(c).cast(pl.Utf8).str.replace_all(r'\D','').str.pad_start(14,'0')
def uf_ok(c):
    return pl.col(c).cast(pl.Utf8).str.strip_chars().str.len_chars() > 0

COLS = ['data_compra','preco_unitario','quantidade','cod_catmat','uf','cnpj_instituicao']
fr=[]
for ano in range(2000, 2026):
    arq = PASTA_BPS / f'BPS_{ano}.parquet'
    if not arq.exists(): continue
    disp = pl.read_parquet(arq, n_rows=0).columns
    fr.append(pl.read_parquet(arq, columns=[c for c in COLS if c in disp]))
bruto = pl.concat(fr, how='diagonal')

mapa = (bruto.filter(uf_ok('uf'))
        .select(n14('cnpj_instituicao').alias('cnpj'),
                pl.col('uf').cast(pl.Utf8).str.strip_chars().alias('u'))
        .group_by(['cnpj','u']).agg(pl.len().alias('n')).sort('n', descending=True)
        .group_by('cnpj').agg(pl.col('u').first().alias('uf_rec')))

ipca = pl.read_parquet(ARQ_IPCA).select(['ano','mes','fator_deflacao'])

b = (bruto.with_columns(n14('cnpj_instituicao').alias('cnpj')).join(mapa, on='cnpj', how='left')
     .with_columns(pl.coalesce([pl.when(uf_ok('uf')).then(pl.col('uf').cast(pl.Utf8).str.strip_chars()),
                                pl.col('uf_rec')]).alias('UF'))
     .filter(pl.col('data_compra').is_not_null()
             & pl.col('data_compra').dt.year().is_between(ANO_INICIO, ANO_FIM)
             & (pl.col('preco_unitario') > 0) & (pl.col('quantidade') > 0)
             & pl.col('cod_catmat').is_not_null())
     .with_columns(pl.col('data_compra').dt.year().alias('ano'),
                   pl.col('data_compra').dt.month().alias('mes'))
     .join(ipca, on=['ano','mes'], how='left'))
assert b.filter(pl.col('fator_deflacao').is_null()).height == 0

b = (b.with_columns((pl.col('preco_unitario')*pl.col('fator_deflacao')).log().alias('lp'),
                    pl.col('quantidade').log().alias('lq'))
       .with_columns(pl.col('lp').median().over(['cod_catmat','ano']).alias('mp'),
                     pl.col('lq').median().over(['cod_catmat','ano']).alias('mq'))
       .with_columns((pl.col('lp')-pl.col('mp')).alias('dev_preco'),
                     (pl.col('lq')-pl.col('mq')).alias('dev_qtd'))
       .with_columns(pl.col('dev_qtd').qcut(5, labels=['Q1','Q2','Q3','Q4','Q5']).alias('quintil')))
print(f'Registros: {b.height:,}')
print(f'Com UF: {b.filter(pl.col("UF").is_not_null()).height/b.height:.1%}')

## 3. Análise 1 e 2 — correlação e quintis de escala

In [ ]:
rs, ps = spearmanr(b['dev_qtd'].to_numpy(), b['dev_preco'].to_numpy())
print(f'Spearman escala x preco: {rs:+.4f} (p={ps:.1e})')

quintis = (b.group_by('quintil')
            .agg(((pl.col('dev_preco').median().exp()-1)*100).round(1).alias('desvio_preco_pct'),
                 pl.len().alias('registros'))
            .sort('quintil'))
print(quintis)

def var_explicada(df, col, alvo='dev_preco'):
    tv, mu = df[alvo].var(), df[alvo].mean()
    g = df.filter(pl.col(col).is_not_null()).group_by(col).agg(pl.col(alvo).mean().alias('m'), pl.len().alias('n'))
    return float((((g['m']-mu)**2)*g['n']).sum()/df.height/tv)

ve_escala = var_explicada(b, 'quintil')
ve_uf     = var_explicada(b, 'UF')
print(f'\nVariancia explicada pela ESCALA: {ve_escala:.2%}')
print(f'Variancia explicada pela UF    : {ve_uf:.2%}')

## 4. Análise 3 — escala típica e preço relativo por UF

In [ ]:
por_uf = (b.filter(pl.col('UF').is_not_null()).group_by('UF')
   .agg(pl.col('dev_qtd').median().exp().round(2).alias('lote_relativo'),
        ((pl.col('dev_preco').median().exp()-1)*100).round(1).alias('desvio_preco_pct'),
        pl.len().alias('registros'))
   .filter(pl.col('registros') >= 500).sort('desvio_preco_pct'))
print(por_uf)

## 5. Análise 4 e 5 — a diferença territorial sobrevive ao controle de escala?
> Compara UFs DENTRO de cada quintil de escala. Em seguida mede quanto a UF
> explica depois de remover a media de cada quintil (residualizacao).

In [ ]:
linhas=[]
for uf in por_uf['UF']:
    s = b.filter(pl.col('UF')==uf)
    d = {'UF': uf}
    for q in ['Q1','Q2','Q3','Q4','Q5']:
        v = s.filter(pl.col('quintil')==q)['dev_preco']
        d[q] = round(float(np.exp(v.median())-1)*100, 1) if v.len() >= 100 else None
    d['bruto'] = round(float(np.exp(s['dev_preco'].median())-1)*100, 1)
    linhas.append(d)
uf_por_quintil = pl.DataFrame(linhas)
print(uf_por_quintil)

resid = b.with_columns((pl.col('dev_preco') - pl.col('dev_preco').mean().over('quintil')).alias('dev_preco'))
ve_uf_ctrl = var_explicada(resid, 'UF')
print(f'\nUF explica {ve_uf:.2%} antes e {ve_uf_ctrl:.2%} depois de controlar a escala')
print(f'Reducao: {(1-ve_uf_ctrl/ve_uf):.0%}')

## 6. Exportar

In [ ]:
resumo = pl.DataFrame({'indicador': ['spearman_escala_preco','var_explicada_escala_pct',
                                     'var_explicada_uf_pct','var_explicada_uf_controlada_pct'],
                       'valor': [round(rs,4), round(ve_escala*100,2),
                                 round(ve_uf*100,2), round(ve_uf_ctrl*100,2)]})
for nome, df in [('escala_quintis', quintis), ('escala_por_uf', por_uf),
                 ('escala_uf_por_quintil', uf_por_quintil), ('escala_resumo', resumo)]:
    df.write_parquet(PASTA_SAIDA / f'{nome}.parquet')
    df.write_csv(PASTA_SAIDA / f'{nome}.csv')
print('=== EXPORTADO ===')
for f in sorted(PASTA_SAIDA.iterdir()): print(' ', f.name)
print('\nValores de referencia para conferencia:')
print('  registros: 883.652 | quantidade preenchida em 100%')
print('  Spearman escala x preco: -0,3458')
print('  variancia explicada: escala 9,97% | UF 2,48% -> 1,88% controlada')
print('  quintis: Q1 +22,4% | Q5 -9,9%')
print('  SC: lote 1,60x, preco -8,4% | PI: lote 0,65x, preco +65,4% | RR: lote 1,99x, preco +37,0%')

## Memória de cálculo — escala e o cruzamento UF x quintil
> Detalha como se chega a cada valor citado, inclusive a tabela cruzada de
> unidade federativa por quintil de escala (nota 79 da revisao).

In [ ]:
print('='*76); print('1. COMO SE DEFINE O QUINTIL DE ESCALA'); print('='*76)
print('  Para cada registro: dev_qtd = log(quantidade) - log(mediana da quantidade')
print('                                 daquele MESMO item naquele MESMO ano)')
print('  Ordenados por dev_qtd, os registros sao divididos em 5 grupos de 20%.')
print(f'\n{"quintil":>8} | {"registros":>10} | {"lote mediano vs item":>21} | {"desvio de preco":>16}')
for r in (b.group_by('quintil').agg(pl.col('dev_qtd').median().exp().alias('lote'),
                                    pl.col('dev_preco').median().exp().alias('p'),
                                    pl.len().alias('n')).sort('quintil').iter_rows(named=True)):
    print(f'{r["quintil"]:>8} | {r["n"]:>10,} | {r["lote"]:>20.2f}x | {(r["p"]-1)*100:>15.1f}%')
print('\n  Leitura: quem compra em lotes menores paga acima da referencia do item.')

In [ ]:
print('='*76); print('2. TABELA CRUZADA — UF POR QUINTIL DE ESCALA (nota 79)'); print('='*76)
print('  Dentro de CADA quintil de escala, calcula-se o desvio de preco de CADA UF.')
print('  Isso responde: comparando compras de tamanho semelhante, as UFs ainda diferem?\n')
print(f'{"UF":>4} | ' + ' | '.join(f'{q:>7}' for q in ['Q1','Q2','Q3','Q4','Q5']) + ' |  geral')
for uf in por_uf['UF']:
    s = b.filter(pl.col('UF') == uf); linha = []
    for q in ['Q1','Q2','Q3','Q4','Q5']:
        v = s.filter(pl.col('quintil') == q)['dev_preco']
        linha.append(f'{(float(np.exp(v.median()))-1)*100:>+6.1f}%' if v.len() >= 100 else '    -  ')
    print(f'{uf:>4} | ' + ' | '.join(linha) + f' | {(float(np.exp(s["dev_preco"].median()))-1)*100:>+6.1f}%')
print('\n  Leitura: a coluna Q5 mostra que, em compras grandes, as UFs convergem.')
print('  A diferenca entre UFs persiste dentro do mesmo quintil -> a escala explica')
print('  parte da diferenca territorial, mas nao toda.')